In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv('SQL_out.csv')
df.info

print(df.shape)
print(df.describe())

In [ ]:
df

In [ ]:
## missing data
total = df.isnull().sum().sort_values(ascending=False)
percent = (df.isnull().sum()/df.isnull().count()).sort_values(ascending=False)
missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
missing_data.head(20)
print(missing_data)
print(df.isnull().sum())

In [ ]:
# Nan
defaultNum = 0
defaultName = 'NaN'

numFeatures = ['age', 'physical_activity', 'sleep']
nameFeatures = ['mental_health_benefits','learning_resources', 'anonymity', 'medical_leave',
                'medical_coverage','mental_health_disorder', 'diagnosed',
                'past_mental_health_disorder','sought_treatment', 'family_history',
                'work_interference', 'gender','race', 'country']

# Cleaning NaN values
for i in df:
    if i in numFeatures:
        df[i] = df[i].fillna(defaultNum)
    elif i in nameFeatures:
        df[i] = df[i].fillna(defaultName)
    else:
        print('Error: Feature %s not recognized.' % i)
df.head(5)

In [ ]:
#clean 'Gender'

print(type('gender'))

#Select unique elements
gender = df['gender'].unique()
#print(df['gender'])

#Made gender groups
male_string = ['male', 'Male', 'Male ', 'Ostensibly Male', 'male, born with xy chromosoms', 'Malel',
               'M', 'MALE', 'm', 'Cis-male', 'Male ', 'cis male', 'Cis Male', 'Man',
               'Cisgender male', 'Let\'s keep it simple and say male""""',
               'Identify as male', 'Masculine', 'Cishet male','I have a penis', 'man',
               'masculino', 'Make', 'CIS Male','mail']
trans_string = ['Trans man', 'transgender', 'Trans woman', 'Trans female']
female_string = ['Female', 'Female ', 'female', 'Woman', 'woman', 'F', 'f', 'I identify as female',
                 '*shrug emoji* (F)', 'Female/gender non-binary.', 'Cis woman',
                 'Female (cisgender)', 'Cis-Female', 'Cisgendered woman', 'She/her/they/them',
                 'Cis female ', 'cisgender female', 'Female-identified', 'cis woman', 'femmina',
                 'Femile', 'Female (cis)']
other_string = ['Agender', 'SWM', 'Genderqueer', 'NaN', '*shrug emoji* (F)', 'Nonbinary', 'Male (or female, or both)',
                'non binary', 'genderfluid' 'Genderqueer', 'genderfluid', 'genderqueer', 'Demiguy', 'none', 'non-binary', 'Other',
                'NB', 'Genderfluid', 'Nonbinary/femme', 'gender non-conforming woman', 'Non-binary',
                'Non binary', 'None', 'agender', 'Questioning', 'rr', 'Agender trans woman', '43',
                'I am a Wookie', 'Trans non-binary/genderfluid', 'Non-binary and gender fluid']

for (row, col) in df.iterrows():
    if col.gender in male_string:
        df['gender'].replace(to_replace = col.gender, value = 'male', inplace = True)
for (row, col) in df.iterrows():
    if col.gender in female_string:
        df['gender'].replace(to_replace = col.gender, value = 'female', inplace = True)
for (row, col) in df.iterrows():
    if col.gender in trans_string:
        df['gender'].replace(to_replace = col.gender, value = 'trans', inplace = True)
for (row, col) in df.iterrows():
    if col.gender in other_string:
        df['gender'].replace(to_replace = col.gender, value = 'other', inplace = True)
        
print(df['gender'])

In [ ]:
# Clean outliers from 'Age'
df = df.drop(df[(df.age < 16)].index)
df.age.unique()

In [ ]:
# Clean 'sought treatment'

# Made groups
one = ['TRUE']
zero = ['FALSE']


for (row, col) in df.iterrows():
    if col.sought_treatment in one:
        df['sought_treatment'].replace(to_replace = col.sought_treatment, value = '1', inplace = True)
for (row, col) in df.iterrows():
    if col.sought_treatment in zero:
        df['sought_treatment'].replace(to_replace = col.sought_treatment, value = '0', inplace = True)
        
print(df['sought_treatment'])

In [ ]:
# clean 'medical coverage'

# Made groups
one = ['TRUE']
zero = ['FALSE']


for (row, col) in df.iterrows():
    if col.medical_coverage in one:
        df['medical_coverage'].replace(to_replace = col.medical_coverage, value = '1', inplace = True)
for (row, col) in df.iterrows():
    if col.medical_coverage in zero:
        df['medical_coverage'].replace(to_replace = col.medical_coverage, value = '0', inplace = True)
        
print(df['medical_coverage'])

In [ ]:
# look at all unique values for every feature
for feature in df:
    print(feature)
    print(df[feature].unique())

In [ ]:
# final check for missing data
total = df.isnull().sum().sort_values(ascending=False)
percent = (df.isnull().sum()/df.isnull().count()).sort_values(ascending=False)
missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
print(missing_data)

In [ ]:
df.shape

In [ ]:
from sklearn import preprocessing

labelDict = {}
for i in df:
    if i != 'age':
        le = preprocessing.LabelEncoder()
        le.fit(df[i])
        le_name_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
        df[i] = le.transform(df[i])
        # Get labels
        labelKey = 'label_' + i
        labelValue = [*le_name_mapping]
        labelDict[labelKey] =labelValue

for key, value in labelDict.items():     
    print(key, value)
df.head()

In [ ]:
# How many people have been treated
import seaborn as sns
plt.figure(figsize=(12,8))
labels = labelDict['label_sought_treatment']
g = sns.countplot(x='sought_treatment', data=df)
g.set_xticklabels(labels)

plt.title('Total Distribution by treated or not')

In [ ]:
# Distribiution and density by Age
import matplotlib.pyplot as plt
plt.figure(figsize=(12,8))
sns.histplot(df["age"], bins=24)
plt.title("Distribution of Participants by Age")
plt.xlabel("Age")

In [ ]:
# Age vs treatment and age vs no treatment
j = sns.FacetGrid(df, col='sought_treatment', height=5)
j = j.map(sns.histplot, "age")

In [ ]:
#correlation matrix
import seaborn as sns
matrix = df.corr()
f, ax = plt.subplots(figsize=(12, 9))
heatmap = sns.heatmap(matrix, vmax=.8, square=True);
heatmap.set_title('Correlation Heatmap', fontdict={'fontsize':12}, pad=12);
plt.show()

In [ ]:
plt.figure(figsize=(8, 12))
heatmap2 = sns.heatmap(df.corr()[['sought_treatment']].sort_values(by='sought_treatment', ascending=False), vmin=-1, vmax=1, annot=True, cmap='BrBG')
heatmap2.set_title('Features Correlating with Willingness to Seek Treatment', fontdict={'fontsize':18}, pad=16);

In [ ]:
# Finding which Machine Learning Model performs the best
# !pip install dabl
import dabl

features = ['mental_health_benefits','family_history','physical_activity','age', 'gender',
                'work_interference','learning_resources','past_mental_health_disorder',
                'medical_coverage','medical_leave','anonymity','sleep', 'mental_health_disorder', 'diagnosed']

x = df[features]
y = df.sought_treatment
X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size = 0.2)
sc = dabl.SimpleClassifier().fit(X_train, Y_train)

In [ ]:
# Logistic Regression

# Define X and Y
feature_cols = ['mental_health_benefits','family_history','physical_activity','age', 'gender',
                'work_interference','learning_resources','past_mental_health_disorder',
                'medical_coverage','medical_leave','anonymity','sleep', 'mental_health_disorder', 'diagnosed', 'work_interference']
x = df[feature_cols]
y = df.sought_treatment

# Scale X
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(x)
X = scaler.transform(x)

xtr, xts, ytr, yts = train_test_split(x, y, test_size=0.35, random_state=0)

# Create Logistic Regression Model
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
df_lr = LogisticRegression(max_iter= 10000)
df_lr.fit(xtr,ytr)
ypred = df_lr.predict(xts)

print('########### Logistic Regression ###############')
confusion = metrics.confusion_matrix(yts, ypred)
TP = confusion[1, 1]
TN = confusion[0, 0]
FP = confusion[0, 1]
FN = confusion[1, 0]    
    
# Precision
print('precision:', metrics.precision_score(yts, ypred))

# Recall
print('recall:', metrics.recall_score(yts, ypred))

# Accuracy
print('accuracy:', metrics.accuracy_score(yts, ypred))

# F1 Score
print('f1 score:', metrics.f1_score(yts, ypred))

# ROC AUC Score
print('ROC AUC score:', metrics.roc_auc_score(yts, ypred))

# Confusion Matrix
from sklearn.metrics import confusion_matrix

sns.heatmap(confusion,annot=True,fmt="d") 
plt.title('confusion matrix')
plt.xlabel('predicted')
plt.ylabel('actual')
plt.show()

# ROC and Area Under the Curve (AUC)
rocauc = metrics.roc_auc_score(yts, ypred)
fpr, tpr, thresholds = metrics.roc_curve(yts, ypred)
plt.figure()
        
plt.plot(fpr, tpr, color='darkorange', label='ROC curve (area = %0.2f)' % rocauc)
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.rcParams['font.size'] = 12
plt.title('ROC curve for treatment classifier')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# K Cross Validation Logisitc Regression
# This checks our model for overfitting
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

df_kf = KFold(n_splits=5)
df_kf

all = cross_val_score(df_lr, xtr, ytr, cv=df_kf)
print('cross validation scores:', all)
print('average:', np.mean(all, axis=None, dtype=None, out=None))

In [ ]:
# Logistic Regression - average scores

num = 100
precision = []
recall = []
accuracy = []
f1Score = []
ROCAUC = []
for i in range(num):
    logreg = LogisticRegression(max_iter= 10000)
    logreg.fit(xtr, ytr)
    ypred = logreg.predict(xts)
    precision.append(metrics.precision_score(yts, ypred))
    recall.append(metrics.recall_score(yts, ypred))
    accuracy.append(metrics.accuracy_score(yts, ypred))
    f1Score.append(metrics.f1_score(yts, ypred))
    ROCAUC.append(metrics.roc_auc_score(yts, ypred))

print('average precision:',np.mean(precision, axis=None, dtype=None, out=None))
print('average recall:',np.mean(recall, axis=None, dtype=None, out=None))
print('average accuracy:',np.mean(accuracy, axis=None, dtype=None, out=None))
print('average f1 score:',np.mean(f1Score, axis=None, dtype=None, out=None))
print('average ROC AUC:',np.mean(ROCAUC, axis=None, dtype=None, out=None))

In [ ]:
# Trying Random Forest Classification
from sklearn.ensemble import RandomForestClassifier
df_rf = RandomForestClassifier(n_estimators = 50)
df_rf.fit(xtr, ytr)
df_rf.score(xts, yts)

In [ ]:
# Trying Support Vector Machine (SVC)
from sklearn.svm import SVC
df_svm = SVC()
df_svm.fit(xtr, ytr)
df_svm.score(xts, yts)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import randint

tree = DecisionTreeClassifier()
featuresSize = feature_cols.__len__()
param_dist = {"max_depth": [3, None],
              "max_features": randint(1, featuresSize),
              "min_samples_split": randint(2, 9),
              "min_samples_leaf": randint(1, 9),
              "criterion": ["gini", "entropy"]}
tree = DecisionTreeClassifier(max_depth=3, min_samples_split=8, max_features=6, criterion='entropy', min_samples_leaf=7)
tree.fit(xtr, ytr)
ypred = tree.predict(xts)


print('########### Decision Tree Classifier ###############')
confusion = metrics.confusion_matrix(yts, ypred)
TP = confusion[1, 1]
TN = confusion[0, 0]
FP = confusion[0, 1]
FN = confusion[1, 0]    
    
# Precision
print('precision:', metrics.precision_score(yts, ypred))

# Recall
print('recall:', metrics.recall_score(yts, ypred))

# Accuracy
print('accuracy:', metrics.accuracy_score(yts, ypred))

# F1 Score
print('f1 score:', metrics.f1_score(yts, ypred))

# ROC AUC Score
print('ROC AUC score:', metrics.roc_auc_score(yts, ypred))

# Confusion Matrix
from sklearn.metrics import confusion_matrix

sns.heatmap(confusion,annot=True,fmt="d") 
plt.title('confusion matrix')
plt.xlabel('predicted')
plt.ylabel('actual')
plt.show()

# ROC and Area Under the Curve (AUC)
rocauc = metrics.roc_auc_score(yts, ypred)
fpr, tpr, thresholds = metrics.roc_curve(yts, ypred)
plt.figure()
        
plt.plot(fpr, tpr, color='darkorange', label='ROC curve (area = %0.2f)' % rocauc)
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.rcParams['font.size'] = 12
plt.title('ROC curve for treatment classifier')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Decision Tree Classifier - average scores

num = 100
precision = []
recall = []
accuracy = []
f1Score = []
ROCAUC = []
for i in range(num):
    tree = DecisionTreeClassifier()
    featuresSize = feature_cols.__len__()
    param_dist = {"max_depth": [3, None],
              "max_features": randint(1, featuresSize),
              "min_samples_split": randint(2, 9),
              "min_samples_leaf": randint(1, 9),
              "criterion": ["gini", "entropy"]}
    tree = DecisionTreeClassifier(max_depth=3, min_samples_split=8, max_features=6, criterion='entropy', min_samples_leaf=7)
    tree.fit(xtr, ytr)
    ypred = tree.predict(xts)
    precision.append(metrics.precision_score(yts, ypred))
    recall.append(metrics.recall_score(yts, ypred))
    accuracy.append(metrics.accuracy_score(yts, ypred))
    f1Score.append(metrics.f1_score(yts, ypred))
    ROCAUC.append(metrics.roc_auc_score(yts, ypred))

print('average precision:',np.mean(precision, axis=None, dtype=None, out=None))
print('average recall:',np.mean(recall, axis=None, dtype=None, out=None))
print('average accuracy:',np.mean(accuracy, axis=None, dtype=None, out=None))
print('average f1 score:',np.mean(f1Score, axis=None, dtype=None, out=None))
print('average ROC AUC:',np.mean(ROCAUC, axis=None, dtype=None, out=None))

In [ ]:
# K Cross Validation Decision Trees
# This checks our model for overfitting
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

df_kf = KFold(n_splits=5)
df_kf

all = cross_val_score(tree, xtr, ytr, cv=df_kf)
print('cross validation scores:', all)
print('average:', np.mean(all, axis=None, dtype=None, out=None))

In [ ]:
# Creating Decision Tree Visualization
#!pip install graphviz
import graphviz
import sklearn.tree as sk
from sklearn import tree
import graphviz
from sklearn.tree import export_graphviz

model=tree.DecisionTreeClassifier(max_depth=3,criterion='entropy')
model.fit(xtr,ytr)

sk.plot_tree(model)
plt.show()

In [ ]:
from graphviz import Source
from sklearn import tree
Source(tree.export_graphviz(model, out_file=None))

In [ ]:
from scipy.stats import t as t_dist
def pairedttest(p):
    phat = np.mean(p)
    n = len(p)
    den = np.sqrt(sum([(diff - phat)**2 for diff in p]) / (n - 1))
    t = (phat * (n**(1/2))) / den
    
    pvalue = t_dist.sf(t, n-1)*2
    return t, pvalue

p1 = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)
for train_index, test_index in kf.split(X):
    xtr, xts, ytr, yts = train_test_split(x, y, test_size=0.35, random_state=0)
    logreg.fit(xtr, ytr)
    tree.fit(xtr, ytr)

    acc1 = metrics.accuracy_score(yts, logreg.predict(xts))
    acc2 = metrics.accuracy_score(yts, tree.predict(xts))
    p1.append(acc1 - acc2)

print("Cross Validated Paired t-test")
t, p = pairedttest(p1)
print(f"t statistic: {t}, p-value: {p}\n")